# Topical Lectures, Controls 
## Drone control, part 4a: test flight

Andreas Freise, Bas Swinkels 27.05.2026

This notebook (and its non-interactive twin `student4b`) is where we put the controllers from the previous notebooks to work. If you can we suggest that you use the one with interactive plots (4a) first. Notebooks `student5` and `student6` then explore alternative, model-based controllers on the same task.

In this notebook, we will try to fly our drone, using the controls we developed in the previous notebooks. To make it a bit more fun, we pose the following challenge: fly your drone through a set of predefined markers and report the time. We will compare the best times at the end of the project.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
import module
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

# interactive plots again
%matplotlib qt

# Don't forget to set the name string to your own name
MyName = "Andreas"

Below we set the calibration constants from notebook 2 and provide the motor-mixing helpers (`set_v`, `set_v_nohover`, `zphi2V`) we have been using throughout.

In [2]:
# Measured parameters for the Crazyflie-scale drone (SI units throughout).
# These would normally come from the student2 system-identification notebook.
V_left_offset = -0.0027   # throttle units; cancels left/right asymmetry
V_hover = 0.5297          # per-motor throttle at hover (= M*g/2 / amp)
V_max = 1.0               # throttle range upper bound
V_min = 0                 # motors are single-direction

# Common-mode / differential decomposition for motor mixing.
# Total motor command must stay in [V_min, V_max]; the differential is
# bounded by the headroom that the common-mode thrust leaves on each side.

def set_v(drone, _V_left, _V_right):
    V_left = V_hover + _V_left + V_left_offset
    V_right = V_hover + _V_right
    fb_z = (V_left + V_right) / 2
    fb_phi = (V_left - V_right) / 2
    fb_z = max(V_min, min(fb_z, V_max))
    headroom = min(fb_z - V_min, V_max - fb_z)
    fb_phi = max(-headroom, min(fb_phi, headroom))
    drone.set_V(fb_z + fb_phi, fb_z - fb_phi)

def set_v_nohover(drone, _V_left, _V_right):
    V_left = _V_left + V_left_offset
    V_right = _V_right
    fb_z = (V_left + V_right) / 2
    fb_phi = (V_left - V_right) / 2
    fb_z = max(V_min, min(fb_z, V_max))
    headroom = min(fb_z - V_min, V_max - fb_z)
    fb_phi = max(-headroom, min(fb_phi, headroom))
    drone.set_V(fb_z + fb_phi, fb_z - fb_phi)

def zphi2V(fb_z, fb_phi):
    V_left = fb_z + fb_phi
    V_right = fb_z - fb_phi
    return V_left, V_right


We use the velocity-loop cascade introduced in notebook 3b: position $\rightarrow$ velocity $\rightarrow$ tilt $\rightarrow$ torque. The five `PIDController` instances and their saturation limits live inside the `Control` class below.

## Racing your drone

In order to test our new control loops we are going to race our drone through a simple track. The goal is to reach a pre-defined set of targets as quickly as possible. In this notebook we will use an interactive plot and keyboard controls to do so.

Below we provide some code that allows keyboard control similar to notebook 1. However, this time the keys do not push the motors directly; instead they shift the set point `(y0, z0)` of the outer position loops. As you will see, flying the drone has become much easier!

In [3]:
class Control():
    def __init__(self, _drone):
        self.drone = _drone
        self.plotter = module.Plotter(self.drone, target_fps=60)

        self.key_dy = 0.05   # m, step per keypress
        self.key_dz = 0.05
        self.stop = False

        # Velocity-loop cascade (same as student3b).
        # Position -> velocity -> tilt -> torque.
        self.y_pos_ctrl = module.PIDController(Kp=-2.0, Ki=0,    Kd=0)
        self.z_pos_ctrl = module.PIDController(Kp=-2.0, Ki=0,    Kd=0)
        self.y_vel_ctrl = module.PIDController(Kp=0.5,  Ki=0.02, Kd=0)
        self.z_vel_ctrl = module.PIDController(Kp=-3.0, Ki=-0.3, Kd=0)
        self.phi_ctrl   = module.PIDController(Kp=0.2,  Ki=0,    Kd=0.04)
        self.v_max_cmd = 1.0                # m/s
        self.tilt_max  = 20 * np.pi / 180   # rad

        self.reset()

    def reset(self):
        self.V_left  = 0
        self.V_right = 0
        self.y0 = 0
        self.z0 = 0
        self.results = np.zeros(12)
        for ctrl in (self.y_pos_ctrl, self.z_pos_ctrl,
                     self.y_vel_ctrl, self.z_vel_ctrl, self.phi_ctrl):
            ctrl.reset()
        self.plotter.reset()

    def physics_step(self):
        """One simulation tick: cascade + drone update + check stop."""
        state = self.drone.update()
        self.results = module.state_to_array(state)

        dy_des = float(np.clip(self.y_pos_ctrl(state.y, state.dy, self.y0),
                               -self.v_max_cmd, self.v_max_cmd))
        dz_des = float(np.clip(self.z_pos_ctrl(state.z, state.dz, self.z0),
                               -self.v_max_cmd, self.v_max_cmd))
        tilt_des = float(np.clip(self.y_vel_ctrl(state.dy, 0, dy_des),
                                 -self.tilt_max, self.tilt_max))
        z_corr = self.z_vel_ctrl(state.dz, 0, dz_des)
        phi_fb = self.phi_ctrl(state.phi, state.dphi, tilt_des)

        hover_ff = V_hover / np.cos(state.phi)
        self.V_left, self.V_right = zphi2V(hover_ff + z_corr, phi_fb)
        set_v_nohover(self.drone, self.V_left, self.V_right)

        if self.results[11] == self.drone.num_targets:
            self.stop = True

    def render_step(self):
        drone_str = f"y0 = {self.y0:.2f}, z0 = {self.z0:.2f}"
        self.plotter.update_display(drone_str)

    def key_control(self):
        self.stop = False
        self.drone.stop = False
        self.plotter.fig.canvas.mpl_disconnect(self.plotter.fig.canvas.manager.key_press_handler_id)
        self.cid = self.plotter.fig.canvas.mpl_connect('key_press_event', self.on_key_simple)

        self.plotter.start_animation(
            self.physics_step,
            self.render_step,
            lambda: self.stop
        )

        self.plotter.fig.canvas.mpl_disconnect(self.cid)

        if self.drone.num_targets > 0:
            if self.results[11] == self.drone.num_targets:
                print(f"All targets reached in: {self.results[0]:.2f} seconds.")
            else:
                print("Not all targets have been reached.")

    def on_key_simple(self, event):
        if event.key == 'escape':
            print('Stopping!')
            self.stop = True
        elif event.key == 'w':
            self.z0 += self.key_dz
        elif event.key == 's':
            self.z0 -= self.key_dz
        elif event.key == 'a':
            self.y0 -= self.key_dy
        elif event.key == 'd':
            self.y0 += self.key_dy


To test our controls we set a 'race' track as a set of targets. 

In [4]:
drone = module.Drone(name=MyName, wind=True)
# Defining the track as an array of targets [y,z,radius].
# The targets have to be reached in the given order.
# The track below is the `offical' track, we want to see your best time for this exact track!
targets = np.array([
    [ 1.0,  1.5, 0.1],
    [ 1.0, -1.5, 0.1],
    [-1.0, -1.5, 0.1],
    [-1.0,  1.5, 0.1],
    [  0,    0, 0.1]
])

# Setting the targets
drone.set_targets(targets)

control = Control(drone)


In [5]:
control.key_control()

All targets reached in: 35.48 seconds.


In [6]:
control.plotter.close()